# Feasibility of a conditional tangent-error bound
### Score-operator error × eigengap × smoothing bias | Existing four-manifold evidence

**Stand-alone Colab; no training or GPU needed for the main analysis.** Uses `FULL_DIAGNOSTIC_TABLE.csv` and `NUMERICAL_QUADRATURE_CONVERGENCE.csv` from our previous independent-seed experiment. Optional final matrix audit reloads one original frozen score model (still no training).

**Scientific objective:** find out whether an upper bound on tangent orientation error can be made **informative** and what *unknown assumptions* prevent certification on real MNIST. The test is deliberately allowed to fail: we report every vacuous, rank-mismatched, quadrature-uncertain, or assumption-violating case.

**Guardrails:** This is NOT a novel Davis–Kahan theorem, NOT a distribution-free MNIST certificate, and NOT a certificate of the original official LHSD algorithm. The current scalar CSV contains a **true-rank eigengap**, not a ground-truth-free estimate of the correct dimension. We use synthetic ground truth ONLY to evaluate the proposed certificate's feasibility.

## Mathematical basis: three separate error mechanisms

At a clean manifold point $x$ and smoothing scale $\sigma$, let

- $P_T(x)$: true geometric rank-$d$ tangent projector (synthetic evaluation only).
- $Q_{\sigma}^{\mathrm{ref}}(x)$: **negative Jacobian of the exact Gaussian-smoothed score**; for the epsilon-normalization in our previous experiment, $Q_{\sigma}^{\mathrm{ref}}=I-\operatorname{Cov}(X\mid X+\sigma Z=x)/\sigma^2$.
- $\widehat Q_{\sigma}(x)=\operatorname{sym}(\sigma J_{\epsilon_\theta}(x,\sigma))$: learned symmetric score operator.
- $P_{\sigma}^{\mathrm{ref}}$: projector onto the $d$ smallest eigenvalues of $Q_{\sigma}^{\mathrm{ref}}$.
- $\widehat P_{\sigma}^{(d)}$: corresponding **oracle-rank** learned eigenspace projector.

Define the operator error and smoothing bias:

$$\varepsilon_\sigma=\|\widehat Q_\sigma-Q_\sigma^{\mathrm{ref}}\|_2,\qquad
 b_\sigma=\frac{\|P_\sigma^{\mathrm{ref}}-P_T\|_F}{\sqrt d}. $$

Let $\widehat\gamma_d=\lambda_{d+1}(\widehat Q_\sigma)-\lambda_d(\widehat Q_\sigma)$ be the eigengap at the **true rank**. A conservative Davis–Kahan/Weyl-style sufficient bound under $\widehat\gamma_d>2\varepsilon_\sigma$ is

$$\frac{\|\widehat P_\sigma^{(d)}-P_T\|_F}{\sqrt d}
\leq \underbrace{\sqrt 2\min\left\{1,\frac{2\varepsilon_\sigma}{\widehat\gamma_d-2\varepsilon_\sigma}\right\}}_{\text{model/eigenspace term}}
+\underbrace{b_\sigma}_{\text{smoothing bias}}.$$

We use a **deliberately conservative coefficient**; tight constants are not the contribution. The triangle inequality and the projector maximum $\sqrt2$ explain the two terms. This is a bound for a *correctly ranked spectral subspace* only. Rank-selection failure is a separate event. See Davis and Kahan (1970), DOI `10.1137/0707001`, and standard Weyl eigenvalue perturbation inequalities.

**Important limitations:** $\varepsilon_\sigma$ and $b_\sigma$ are not observables on MNIST. The previous CSV reports $\|\widehat Q-Q^{\mathrm{ref}}_{\rm grid}\|_F/\sqrt8$; we multiply by $\sqrt8$ to obtain a conservative proxy upper bound on its *grid-reference* operator norm. A finite quadrature grid is not a rigorously certified exact score, and finite-difference approximation is not bounded here. The notebook checks numerical convergence and calls these **reference-conditional diagnostic bounds**, not verified global certificates.

In [ ]:
# 0. Load saved evidence; no retraining or Colab GPU required.
import os, json, math, warnings, zipfile
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import spearmanr
D=8
SQRT2=np.sqrt(2.)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive')
except ImportError:
    ROOT=Path('/mnt/data') if Path('/mnt/data').exists() else Path.cwd()

INPUT_OVERRIDE=''  # Optional: absolute path of FULL_DIAGNOSTIC_TABLE.csv
QUAD_OVERRIDE=''   # Optional: absolute path of NUMERICAL_QUADRATURE_CONVERGENCE.csv
OUTPUT_DIR=ROOT/'Spectral_Error_Bound_Feasibility'
OUT=Path(OUTPUT_DIR);OUT.mkdir(parents=True,exist_ok=True)

def locate(filename,override='',directories=None):
    candidates=[]
    if override:candidates.append(Path(override))
    for p in (directories or []):candidates.append(Path(p)/filename)
    candidates += [ROOT/filename,Path('/content')/filename,Path('/mnt/data')/filename]
    for p in candidates:
        if p.exists():print(filename,'<-',p);return p
    raise FileNotFoundError(f'Missing {filename}. Upload to Colab or set INPUT_OVERRIDE/QUAD_OVERRIDE. Tried: {candidates}')

SOURCEDIR=ROOT/'Geometry_Reliability_Followup_Independent_Seeds'
p=locate('FULL_DIAGNOSTIC_TABLE.csv',INPUT_OVERRIDE,[SOURCEDIR])
FULL=pd.read_csv(p)
assert len(FULL)>0
requirements={'family','model_id','split','point_id','sigma','true_rank','estimated_rank',
 'oracle_rank_tangent_error','estimated_rank_tangent_error','score_eigenspace_component',
 'smoothing_component','score_operator_error','true_rank_eigengap','rank_component','dimension_MAE'}
assert requirements<=set(FULL.columns),f'Missing columns: {requirements-set(FULL.columns)}'
FAMILIES=sorted(FULL.family.unique().tolist())
print('Data:',FULL.shape,'families:',FAMILIES,'splits:',FULL.split.value_counts().to_dict())
print('Number of independent model IDs:',FULL.model_id.nunique())
display(FULL.groupby(['family','split']).agg(n_models=('model_id','nunique'),n_points=('point_id','nunique')).reset_index())

In [ ]:
# 1. Audit numerical reference convergence; do not silently certify un-converged rows.
qfile=locate('NUMERICAL_QUADRATURE_CONVERGENCE.csv',QUAD_OVERRIDE,[SOURCEDIR])
QUAD=pd.read_csv(qfile)
assert {'family','sigma','pass_tolerance','max_relative_Q_grid_error'}<=set(QUAD.columns)
QUAD.pass_tolerance=QUAD.pass_tolerance.astype(str).str.lower().eq('true')
print('Grid comparisons passing:',int(QUAD.pass_tolerance.sum()),'/',len(QUAD))
display(QUAD)
# A converged finite-grid comparison is a numerical diagnostic, not a rigorous quadrature error bound.
QUAD.to_csv(OUT/'REFERENCE_GRID_AUDIT.csv',index=False)

## Experiment A — Oracle-reference feasibility of a Davis–Kahan bound

The first test uses **only previously saved matrices summarized in CSV**. We do not claim that an MNIST model can compute $\varepsilon$ without an independently validated reference.

The available `score_operator_error` gives a Frobenius-norm discrepancy to the numerical smoothed-score reference. Hence $\delta_F=\sqrt{8}\,\mathrm{score\_operator\_error}$ is an upper bound on the spectral norm of the discrepancy *to that numerical reference*. We use it conservatively. `true_rank_eigengap` is a learned-operator gap **at oracle rank**, not a rank-free field estimate.

We mark a bound **informative** only when the model/eigenspace term plus smoothing bias is strictly below $\sqrt2$, the trivial maximum normalized error of two rank-$d$ projectors.

In [ ]:
# 2. Compute per-point feasibility and verify triangle inequalities.
T=FULL[FULL.split=='test'].copy()
assert not T.duplicated(['family','model_id','point_id','sigma']).any()
T=T.merge(QUAD[['family','sigma','pass_tolerance','max_relative_Q_grid_error']],
          on=['family','sigma'],how='left',validate='many_to_one')
assert T.pass_tolerance.notna().all()
T['rank_correct']=(T.estimated_rank==T.true_rank)
T['delta_F_grid']=np.sqrt(D)*T.score_operator_error
T['gap_hat_oracle_rank']=T.true_rank_eigengap.clip(lower=0)
T['ratio_delta_to_gap']=T.delta_F_grid/T.gap_hat_oracle_rank.replace(0,np.nan)
T['gap_condition']=(T.gap_hat_oracle_rank>2*T.delta_F_grid)
valid=T.gap_condition
T['dk_model_term']=np.where(valid,SQRT2*np.minimum(1.,
    2*T.delta_F_grid/(T.gap_hat_oracle_rank-2*T.delta_F_grid).where(valid,np.nan)),np.nan)
T['bound_oracle_rank']=T.dk_model_term+T.smoothing_component
T['bound_informative']=valid & (T.bound_oracle_rank < SQRT2-1e-9)
T['eligible_reference_conditional']=T.rank_correct & T.bound_informative & T.pass_tolerance
T['orientation_bound_holds']=np.where(valid,
    T.oracle_rank_tangent_error<=T.bound_oracle_rank+1e-7,np.nan)
T['model_component_bound_holds']=np.where(valid,
    T.score_eigenspace_component<=T.dk_model_term+1e-7,np.nan)
T['triangle_holds']=T.estimated_rank_tangent_error <= (
    T.rank_component+T.score_eigenspace_component+T.smoothing_component+1e-8)
assert T.triangle_holds.all(), 'Unexpected failure: stored decomposition violates triangle inequality'
assert T.loc[valid,'orientation_bound_holds'].all(), 'Investigate DK bound derivation or numerical data consistency'
assert T.loc[valid,'model_component_bound_holds'].all()
T.to_csv(OUT/'POINTWISE_FEASIBILITY.csv',index=False)

S=T.groupby(['family','sigma']).agg(
 n=('point_id','size'),rank_correct_rate=('rank_correct','mean'),
 gap_condition_rate=('gap_condition','mean'),
 informative_rate=('bound_informative','mean'),
 eligible_and_reference_converged_rate=('eligible_reference_conditional','mean'),
 mean_oracle_rank_error=('oracle_rank_tangent_error','mean'),
 mean_deployable_rank_error=('estimated_rank_tangent_error','mean'),
 median_delta_F_grid=('delta_F_grid','median'),
 median_learned_gap_at_oracle_rank=('gap_hat_oracle_rank','median'),
 mean_smoothing_bias=('smoothing_component','mean'),
 grid_pass=('pass_tolerance','all')).reset_index()
S.to_csv(OUT/'FEASIBILITY_BY_FAMILY_SIGMA.csv',index=False)
print('Reference-conditional bounds: if gap condition holds, inequality checks pass.')
print('How often is the bound informative rather than automatically true?')
display(S.round(5))

In [ ]:
# 3. Plot how close the score approximation is to the eigengap threshold.
fig,ax=plt.subplots(figsize=(9,5.5))
for fam,group in T.groupby('family'):
    x=group.gap_hat_oracle_rank.to_numpy()
    y=(2*group.delta_F_grid).to_numpy()
    ax.scatter(x,y,s=6,alpha=.18,label=fam)
lim=max(np.nanquantile(T.gap_hat_oracle_rank,.995),np.nanquantile(2*T.delta_F_grid,.995))
ax.plot([0,lim],[0,lim],ls='--',lw=2,c='black',label='gap = 2δ (feasibility threshold)')
ax.set(xlim=(0,lim),ylim=(0,lim),xlabel='Learned eigengap at true synthetic rank',
       ylabel='2 × operator discrepancy upper bound')
ax.set_title('Can the spectral separation dominate learned score error?')
ax.legend(loc='best');fig.tight_layout()
fig.savefig(OUT/'FigA_Eigengap_vs_Operator_Error.png',dpi=180);plt.show()

In [ ]:
# 4. Non-vacuous rate and bound-vs-observed accuracy, by manifold.
fs=sorted(T.family.unique()); fig,axes=plt.subplots(1,2,figsize=(12,4.5))
vals=[]
for f in fs:
 x=T[T.family==f]
 vals.append([x.rank_correct.mean(),x.gap_condition.mean(),x.bound_informative.mean(),
              x.eligible_reference_conditional.mean()])
v=np.array(vals); xx=np.arange(len(fs));w=.20
for j,name in enumerate(['rank matches','spectral gap passes','non-vacuous','usable with grid check']):
 axes[0].bar(xx+(j-1.5)*w,v[:,j],width=w,label=name)
axes[0].set_xticks(xx,fs);axes[0].set_ylim(0,1)
axes[0].set_title('Fraction of candidate estimates'); axes[0].legend(fontsize=8)
for f,g in T[T.bound_informative].groupby('family'):
 axes[1].scatter(g.oracle_rank_tangent_error,g.bound_oracle_rank,s=9,alpha=.35,label=f)
axes[1].plot([0,SQRT2],[0,SQRT2],ls='--',c='black',lw=1)
axes[1].set(xlim=(0,SQRT2),ylim=(0,SQRT2),xlabel='Observed oracle-rank tangent error',
 ylabel='Conditional bound (grid reference)',title='A valid but loose upper bound?')
axes[1].legend(fontsize=8);fig.tight_layout()
fig.savefig(OUT/'FigB_Conditional_Bound_Utility.png',dpi=180);plt.show()

## Experiment B — Smoothing bias and learned-score error are different

The true tangent of the original manifold and the tangent eigenspace of the *Gaussian-smoothed* distribution need not coincide, even if the learned network is perfect. On the other hand, a weak network can have large $\|\widehat Q-Q^{\mathrm{ref}}\|$ even when smoothing bias is negligible. They must not be conflated.

Both components here use synthetic oracle knowledge. We analyze their empirical variation across $\sigma$, model seeds and manifold families and check whether a simple bias envelope fitted on FIT/CAL generalizes. **This is diagnostic feasibility, not a new theorem.**

In [ ]:
# 5. Quantify two competing mechanisms and model-seed dispersion.
GG=T.groupby(['family','model_id','sigma']).agg(
  score_err=('score_operator_error','mean'),bias=('smoothing_component','mean'),
  gap=('gap_hat_oracle_rank','mean'),e_oracle=('oracle_rank_tangent_error','mean')).reset_index()
G=GG.groupby(['family','sigma']).agg(
  score_mean=('score_err','mean'),score_model_sd=('score_err','std'),
  smoothing_mean=('bias','mean'),smoothing_model_sd=('bias','std'),
  eigengap_mean=('gap','mean'),tangent_mean=('e_oracle','mean')).reset_index()
G.to_csv(OUT/'MODEL_ERROR_VS_SMOOTHING.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for fam,g in G.groupby('family'):
 g=g.sort_values('sigma')
 axes[0].plot(g.sigma,g.score_mean,'o-',label=fam)
 axes[1].plot(g.sigma,g.smoothing_mean,'o-',label=fam)
axes[0].set(xlabel='Gaussian noise σ',ylabel='Mean Frobenius score-operator discrepancy / √8',title='Learned-score approximation')
axes[1].set(xlabel='Gaussian noise σ',ylabel='Mean reference-vs-true tangent discrepancy',title='Noise-induced smoothing bias')
axes[0].legend();axes[1].legend();fig.tight_layout()
fig.savefig(OUT/'FigC_Model_Error_vs_Smoothing_Bias.png',dpi=180);plt.show()
display(G.round(5))

## Experiment C — Can the bound become usable **without target tangent labels**?

To apply a conditional spectral bound, we need an externally justified number $\delta$ such that $\|\widehat Q-Q^{\rm ref}\|_2\le\delta$, and a justified bound $b\le B$ on smoothing bias. Neither follows from small projector defect, score-model agreement, or denoiser consistency alone.

We compare three deliberately labeled scenarios:

1. **Oracle-reference upper bound**: uses the exact synthetic discrepancy and bias per point. Useful only to diagnose mathematical feasibility.
2. **Synthetic FIT/CAL envelopes**: estimate $\delta$ and $B$ using reference data in FIT/CAL; evaluate assumptions and inequality on untouched TEST, but **no finite-sample deterministic guarantee** is implied by the sample maxima.
3. **Cross-manifold envelope transfer**: fit envelope on one source family and apply it to another, explicitly testing assumption violations. This exposes why a bound cannot simply be exported to real MNIST.

Do not use test error labels to choose envelopes or thresholds.

In [ ]:
# 6. Calibrate *empirical envelopes* on prior FIT+CAL only (oracle supervised), test separately.
TRAIN=FULL[FULL.split.isin(['fit','cal'])].copy()
TRAIN['delta_F_grid']=np.sqrt(D)*TRAIN.score_operator_error
ENV=(TRAIN.groupby(['family','sigma']).agg(
  delta_fitcal_max=('delta_F_grid','max'),
  bias_fitcal_max=('smoothing_component','max'),
  n_training_rows=('point_id','size')).reset_index())
ENV.to_csv(OUT/'SYNTHETIC_FITCAL_ENVELOPES.csv',index=False)
V=T.merge(ENV,on=['family','sigma'],validate='many_to_one')
V['operator_assumption_holds']=V.delta_F_grid <= V.delta_fitcal_max+1e-12
V['bias_assumption_holds']=V.smoothing_component <= V.bias_fitcal_max+1e-12
V['both_assumptions_hold']=V.operator_assumption_holds & V.bias_assumption_holds
V['envelope_gap_ok']=V.gap_hat_oracle_rank > 2*V.delta_fitcal_max
V['envelope_bound']=np.where(V.envelope_gap_ok,
 SQRT2*np.minimum(1.,2*V.delta_fitcal_max/
 (V.gap_hat_oracle_rank-2*V.delta_fitcal_max).where(V.envelope_gap_ok,np.nan)) + V.bias_fitcal_max,np.nan)
V['envelope_informative']=V.envelope_gap_ok & (V.envelope_bound<SQRT2-1e-9)
V['envelope_conditional_pass']=np.where(V.envelope_gap_ok & V.both_assumptions_hold,
 V.oracle_rank_tangent_error<=V.envelope_bound+1e-7,np.nan)
V.to_csv(OUT/'FITCAL_ENVELOPE_ON_TEST.csv',index=False)
rows=[]
for fam,g in V.groupby('family'):
 condition=g.envelope_gap_ok & g.both_assumptions_hold
 rows.append(dict(family=fam,n=len(g),envelope_operator_hold=g.operator_assumption_holds.mean(),
   envelope_smoothing_hold=g.bias_assumption_holds.mean(),both_assumptions_hold=g.both_assumptions_hold.mean(),
   bound_gap_pass=g.envelope_gap_ok.mean(),bound_nonvacuous=g.envelope_informative.mean(),
   conditional_violations=int((g.loc[condition,'envelope_conditional_pass']==False).sum()),
   informative_and_all_assumptions=(g.envelope_informative&g.both_assumptions_hold&g.rank_correct).mean()))
ENV_SUM=pd.DataFrame(rows)
ENV_SUM.to_csv(OUT/'FITCAL_ENVELOPE_FEASIBILITY.csv',index=False)
display(ENV_SUM.round(5))
print('Fitting/calibration envelopes use synthetic ground-truth reference: NOT deployable on unknown MNIST geometry.')

In [ ]:
# 7. Cross-family stress: apply SOURCE envelope to TARGET and show violation of its assumptions.
rows=[]
for source in FAMILIES:
 source_env=ENV.loc[ENV.family==source,['sigma','delta_fitcal_max','bias_fitcal_max']]
 x=T.merge(source_env,on='sigma',how='inner',validate='many_to_one')
 x['assumption']= (x.delta_F_grid<=x.delta_fitcal_max+1e-12)&(x.smoothing_component<=x.bias_fitcal_max+1e-12)
 x['gap_ok']=x.gap_hat_oracle_rank>2*x.delta_fitcal_max
 x['bound']=np.where(x.gap_ok,
   SQRT2*np.minimum(1.,2*x.delta_fitcal_max/(x.gap_hat_oracle_rank-2*x.delta_fitcal_max).where(x.gap_ok,np.nan))
     +x.bias_fitcal_max,np.nan)
 for target,g in x.groupby('family'):
  rows.append(dict(source_family=source,target_family=target,n=len(g),
       assumed_bounds_hold=g.assumption.mean(),gap_pass=g.gap_ok.mean(),
       informative=(g.gap_ok & (g.bound<SQRT2-1e-9)).mean(),
       usable_if_rank_known=(g.gap_ok & (g.bound<SQRT2-1e-9) & g.rank_correct & g.assumption).mean()))
TRANSFER=pd.DataFrame(rows)
TRANSFER.to_csv(OUT/'CROSS_FAMILY_ENVELOPE_STRESS.csv',index=False)
pivot=TRANSFER.pivot(index='source_family',columns='target_family',values='assumed_bounds_hold')
fig,ax=plt.subplots(figsize=(6,5));im=ax.imshow(pivot.values,vmin=0,vmax=1,cmap='viridis')
ax.set_xticks(range(len(pivot.columns)),pivot.columns);ax.set_yticks(range(len(pivot.index)),pivot.index)
ax.set(xlabel='Target family',ylabel='Envelope-fitting family',title='Fraction of TEST points satisfying both assumptions')
for i in range(len(pivot.index)):
 for j in range(len(pivot.columns)):
  ax.text(j,i,f'{pivot.iloc[i,j]:.0%}',ha='center',va='center',color='white' if pivot.iloc[i,j]<.55 else 'black')
fig.colorbar(im,ax=ax);fig.tight_layout()
fig.savefig(OUT/'FigD_Cross_Family_Assumption_Transfer.png',dpi=180);plt.show()
display(TRANSFER.round(3))

In [ ]:
# 8. Hypothetical assumed operator-error budgets: what δ would make a non-vacuous bound possible?
# Assumed deltas are *NOT inferred* from a model. This is an identifiability/feasibility curve.
GRID_DELTA=[0.01,0.02,0.04,0.08,0.12,0.20,0.32,0.50,0.80]
rows=[]
for family,g in T.groupby('family'):
 for delta in GRID_DELTA:
  passgap=g.gap_hat_oracle_rank>2*delta
  model_term=np.where(passgap,SQRT2*np.minimum(1.,
      2*delta/(g.gap_hat_oracle_rank-2*delta).where(passgap,np.nan)),np.nan)
  # For plot purposes B is measured synthetic smoothing bias; not available on MNIST.
  informative=passgap & ((model_term+g.smoothing_component)<SQRT2-1e-9)
  assumption=g.delta_F_grid<=delta
  rows.append(dict(family=family,assumed_delta=delta,n=len(g),
      actual_score_error_below_assumed_delta=assumption.mean(),
      spectral_gap_condition_rate=passgap.mean(),
      nonvacuous_with_oracle_smoothing=informative.mean(),
      condition_and_nonvacuous_and_correct_rank=(assumption&informative&g.rank_correct).mean()))
SEN=pd.DataFrame(rows);SEN.to_csv(OUT/'ASSUMED_OPERATOR_BUDGET_SENSITIVITY.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for fam,g in SEN.groupby('family'):
 axes[0].plot(g.assumed_delta,g.nonvacuous_with_oracle_smoothing,'o-',label=fam)
 axes[1].plot(g.assumed_delta,g.condition_and_nonvacuous_and_correct_rank,'o-',label=fam)
axes[0].set(xlabel='Assumed operator error δ (absolute spectral units)',ylabel='Fraction with non-vacuous formula',title='Feasibility assuming δ is justified')
axes[1].set(xlabel='Assumed operator error δ',ylabel='Fraction actual error ≤ δ, rank correct, non-vacuous',title='Tradeoff: assumption truth versus certificate strength')
axes[0].legend();axes[1].legend();fig.tight_layout()
fig.savefig(OUT/'FigE_Operator_Error_Assumption_Sensitivity.png',dpi=180);plt.show()

## Optional matrix audit — exact spectral norm rather than Frobenius upper bound

The main analysis already answers the feasibility question using saved outputs and costs seconds. This optional audit **does not retrain**. It reloads the original model for a small number of points, recomputes the learned symmetric Jacobian and the Gaussian smoothed-score reference by deterministic quadrature, and compares:

- Exact grid-reference $\|\widehat Q-Q^{\mathrm{ref}}_{\rm grid}\|_2$ versus the existing conservative Frobenius bound.
- Eigengaps of learned and reference matrices.
- Empirical projector rotation versus the theoretical conditional bound.
- Difference between 256-point and 512-point quadrature grids (the latter is a convergence diagnostic, **not** a rigorous numerical error guarantee).

Run it only if you have your previous frozen checkpoint in Drive. Set `RUN_MATRIX_AUDIT=True`; default is **False** so the basic experiment is immediately executable even without the checkpoints. The audit uses the same original manifold parametrization as the previous experiment.

In [ ]:
# 9. OPTIONAL: reload original frozen synthetic epsilon network and calculate actual operator norm.
RUN_MATRIX_AUDIT=False
AUDIT_FAMILIES=['torus','circle']
AUDIT_SIGMAS=[0.03,0.128,0.342]
AUDIT_N=6
AUDIT_REFERENCE_GRID=256
AUDIT_CHECK_512=False  # set True to check finer grid (torus 512^2 may require CPU time)

if RUN_MATRIX_AUDIT:
 import torch
 from torch import nn
 from scipy.special import logsumexp
 D=8
 DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
 SEED_GEOMETRY=73
 b0=np.random.default_rng(SEED_GEOMETRY).normal(size=(D,2));B,_=np.linalg.qr(b0)
 v0=np.random.default_rng(SEED_GEOMETRY+101).normal(size=D)
 v0=v0-B@(B.T@v0);v0/=np.linalg.norm(v0)
 B3=np.column_stack([B,v0]);R_MAJOR=1.20;R_MINOR=.45
 class EpsNetwork(nn.Module):
  def __init__(self,D=8,width=128):
   super().__init__();self.net=nn.Sequential(nn.Linear(D+1,width),nn.SiLU(),nn.Linear(width,width),nn.SiLU(),nn.Linear(width,width),nn.SiLU(),nn.Linear(width,D))
  def forward(self,y,sigma):
   if not isinstance(sigma,torch.Tensor):sig=torch.full((len(y),1),float(sigma),device=y.device,dtype=y.dtype)
   else:sig=sigma
   return self.net(torch.cat([y,torch.log(sig)],dim=1))
 def pts_and_tangents(fam,p):
  if fam=='circle':
   t=p[:,0]; X=np.cos(t)[:,None]*B[:,0]+np.sin(t)[:,None]*B[:,1]
   U=(-np.sin(t)[:,None]*B[:,0]+np.cos(t)[:,None]*B[:,1])[:,None,:]
   return X,U
  if fam=='torus':
   u,v=p[:,0],p[:,1];r=R_MAJOR+R_MINOR*np.cos(v)
   X=np.stack([r*np.cos(u),r*np.sin(u),R_MINOR*np.sin(v)],1)@B3.T
   U=np.stack([-np.sin(u),np.cos(u),np.zeros_like(u)],1)@B3.T
   V=np.stack([-np.sin(v)*np.cos(u),-np.sin(v)*np.sin(u),np.cos(v)],1)@B3.T
   return X,np.stack([U,V],1)
  raise ValueError('Matrix audit currently supports only circle and torus')
 def quad_reference(fam,Y,sigma,res):
  a=(np.arange(res)+.5)*2*np.pi/res
  if fam=='circle':params=a[:,None]
  else:
   u,v=np.meshgrid(a,a,indexing='ij');params=np.stack([u.ravel(),v.ravel()],1)
  XX,_=pts_and_tangents(fam,params)
  out=[]
  for y in Y:
   d2=np.sum((XX-y)**2,axis=1)
   w=np.exp(-d2/(2*sigma*sigma)-logsumexp(-d2/(2*sigma*sigma)))
   mu=w@XX
   C=np.einsum('m,mi,mj->ij',w,XX,XX)-np.outer(mu,mu)
   out.append(np.eye(D)-C/(sigma*sigma))
  return np.stack(out)
 @torch.no_grad()
 def learned_q(model,Y,sigma,h=.001):
  y=torch.as_tensor(Y,dtype=torch.float32,device=DEVICE)
  ans=[]
  for j in range(D):
   yp=y.clone();ym=y.clone();yp[:,j]+=h;ym[:,j]-=h
   ans.append(((model(yp,sigma)-model(ym,sigma))/(2*h)*sigma).cpu().numpy())
  Q=np.stack(ans,axis=-1).astype(np.float64)
  return (Q+Q.transpose(0,2,1))/2
 def prj(Q,d):
  ev,vec=np.linalg.eigh(Q);return vec[:,:,:d]@vec[:,:,:d].transpose(0,2,1),ev
 paths={'circle':ROOT/'synthetic_tangent_ground_truth'/'synthetic_eps_network.pt'}
 for f in AUDIT_FAMILIES:
  if f!='circle':paths[f]=ROOT/'synthetic_tangent_ground_truth'/f'{f}_eps_R8_seed73_steps2500_v1.pt'
 audit=[]
 for fam in AUDIT_FAMILIES:
  path=paths[fam]
  if not path.exists():
   print('SKIP missing frozen checkpoint:',path);continue
  model=EpsNetwork().to(DEVICE)
  model.load_state_dict(torch.load(path,map_location=DEVICE,weights_only=True));model.eval()
  for param in model.parameters():param.requires_grad_(False)
  rng=np.random.default_rng(20261108)
  params=rng.uniform(0,2*np.pi,size=(AUDIT_N,1 if fam=='circle' else 2))
  X,U=pts_and_tangents(fam,params);d=U.shape[1]
  Ptrue=U.transpose(0,2,1)@U
  for sigma in AUDIT_SIGMAS:
   Qhat=learned_q(model,X,sigma)
   Qref=quad_reference(fam,X,sigma,AUDIT_REFERENCE_GRID)
   if AUDIT_CHECK_512:
    fine=quad_reference(fam,X,sigma,512)
    relative_grid=np.linalg.norm(Qref-fine,axis=(1,2))/(np.linalg.norm(fine,axis=(1,2))+1e-12)
   else: relative_grid=np.full(len(X),np.nan)
   Ph,eh=prj(Qhat,d);Pr,er=prj(Qref,d)
   delta_op=np.linalg.norm(Qhat-Qref,ord=2,axis=(1,2))
   delta_f=np.linalg.norm(Qhat-Qref,axis=(1,2))
   gap_h=eh[:,d]-eh[:,d-1];gap_ref=er[:,d]-er[:,d-1]
   for i in range(len(X)):
    denom=np.sqrt(d);e_model=np.linalg.norm(Ph[i]-Pr[i])/denom
    bias=np.linalg.norm(Pr[i]-Ptrue[i])/denom
    def bound(delta):
     return SQRT2*min(1,2*delta/(gap_h[i]-2*delta)) if gap_h[i]>2*delta else np.nan
    audit.append(dict(family=fam,index=i,sigma=sigma,delta_operator=delta_op[i],delta_F=delta_f[i],
     learned_gap=gap_h[i],reference_gap=gap_ref[i],model_projector_error=e_model,
     smoothing_bias=bias,total_tangent_error=np.linalg.norm(Ph[i]-Ptrue[i])/denom,
     bound_using_operator_norm=bound(delta_op[i]),bound_using_Frobenius=bound(delta_f[i]),
     grid_relative_difference=relative_grid[i]))
 if audit:
  M=pd.DataFrame(audit);M.to_csv(OUT/'OPTIONAL_MATRIX_SPECTRAL_NORM_AUDIT.csv',index=False)
  display(M.groupby(['family','sigma']).agg(delta_op=('delta_operator','mean'),
   learned_gap=('learned_gap','mean'),bound_op=('bound_using_operator_norm','mean'),
   bound_f=('bound_using_Frobenius','mean'),actual_model_error=('model_projector_error','mean')).round(4))
  valid=M.bound_using_operator_norm.notna()
  assert (M.loc[valid,'model_projector_error']<=M.loc[valid,'bound_using_operator_norm']+1e-6).all()
  print('Optional matrix audit: valid conditional DK inequalities confirmed on computed matrices')
 else:print('No checkpoints found; CSV-only analysis is complete.')
else:print('Matrix audit skipped (set RUN_MATRIX_AUDIT=True to run from frozen checkpoints).')

## Interpretation and next research contribution

A **useful conditional guarantee** would require three pieces:

1. **Correct rank / reliable rank identification** (otherwise the rank-$d$ subspace theorem does not control the deployable output).
2. An *external, justified* bound $\|\widehat Q-Q^{\rm ref}\|_2\le\delta$ which is small relative to an appropriate eigengap. This is a theorem assumption, **not a quantity estimated by nonconservativity or ensemble agreement without validation**.
3. An independent bound on the distance between the Gaussian-smoothed reference eigenspace and the true manifold tangent. Curvature/reach/smoothing assumptions might make this possible, but we have **not proved such an estimate**.

If these requirements are rarely satisfied (or the resulting error upper bound is close to the trivial $\sqrt2$), Davis–Kahan alone will not give a practical certificate. That negative result would directly motivate stronger *structure-aware* perturbation bounds, locally verifiable score-error controls, or assumption-conditional empirical risk assessment.

### Strong proposal framing

> We empirically decompose tangent-estimation error into learned-score operator perturbations, eigenspace instability, noise-induced smoothing bias, and rank-selection errors. We investigate conditions under which operator-error bounds and eigengaps lead to informative tangent-orientation guarantees, and identify why agreement-based proxies alone cannot certify such geometry on real data.

Never claim the FIT/CAL maxima are rigorous distribution-free bounds, or that synthetic oracle references are available for MNIST.

In [ ]:
# 10. Export precise evidence and negative findings, including any inconclusive reference grids.
by_fam=T.groupby('family').agg(
  rows=('point_id','size'),models=('model_id','nunique'),
  rank_correct=('rank_correct','mean'),gap_condition=('gap_condition','mean'),
  nonvacuous=('bound_informative','mean'),
  nonvacuous_rank_correct_grid_pass=('eligible_reference_conditional','mean'),
  learned_gap_median=('gap_hat_oracle_rank','median'),
  fro_error_upper_median=('delta_F_grid','median'),
  mean_true_oracle_rank_error=('oracle_rank_tangent_error','mean'),
  mean_true_estimated_rank_error=('estimated_rank_tangent_error','mean')).reset_index()
by_fam.to_csv(OUT/'PROPOSAL_FEASIBILITY_SUMMARY.csv',index=False)
lines=['# Spectral tangent-error bound feasibility — Research evidence',
'', '## Conditional formula',
'Under correct intrinsic rank, a suitable spectral separation condition, an independently justified score-operator error bound delta, and an upper bound B on smoothed-manifold bias:',
'normalized tangent-projector error <= sqrt(2)*min(1,2*delta/(gap-2*delta))+B, provided gap>2*delta.',
'', 'This uses standard spectral perturbation results; it is not a new theorem.',
'', '## Computed synthetic diagnostic results',by_fam.round(5).to_markdown(index=False),
'', '## Critical limitations',
'- Operator discrepancy is to a NUMERICAL reference constructed using the synthetic data-generating distribution.',
'- Operator norm is upper-bounded by Frobenius discrepancy / sqrt(D) * sqrt(D); conservative and potentially vacuous.',
'- The eigengap was evaluated at the TRUE synthetic rank; real MNIST rank is unknown.',
'- Gaussian-smoothed reference projector is not necessarily the original data tangent.',
'- Finite quadrature convergence is diagnostic, not a rigorous global integration error guarantee.',
'- FIT/CAL maxima are empirical envelopes and can fail under distribution shift.',
'- No real-MNIST tangent-error coverage certificate follows from these tests.',
'', '## Proposed mathematical aims',
'1. Derive verified bounds on the learned score-operator approximation using observable or structurally justified error information.',
'2. Bound the noise-dependent bias of smoothed-score geometry under reach/curvature/density assumptions.',
'3. Establish a rank-aware, gap-aware abstention criterion with non-vacuous coverage.',
'4. Replicate on independent score models/manifold families and quantify where conditional assumptions fail.',
'', '## Numerical reference audit',QUAD.to_markdown(index=False),
'', '## Source', 'User experimental outputs: FULL_DIAGNOSTIC_TABLE.csv and NUMERICAL_QUADRATURE_CONVERGENCE.csv.']
(OUT/'PROPOSAL_ERROR_BOUND_EVIDENCE.md').write_text('\n'.join(lines).replace('\\n','\n'))
protocol=dict(input_full=str(p),input_quadrature=str(qfile),
  dataset_rows=len(FULL),families=FAMILIES,
  oracle_reference_only=True,real_data_certificate=False,rank_oracle_for_gap=True,
  main_formula='sqrt(2)*min(1,2*delta/(gap-2*delta))+smoothing_bias, gap>2*delta',
  caveats=['grid accuracy is not certified by coarse/fine comparison','fit/cal maxima are NOT deterministic bounds',
           'no ground-truth geometry for MNIST','rank mismatch requires separate analysis',
           'score_error upper bound uses Frobenius norm'])
(OUT/'PROTOCOL.json').write_text(json.dumps(protocol,indent=2))
print('OUTPUT:',OUT)
print('Report:',OUT/'PROPOSAL_ERROR_BOUND_EVIDENCE.md')
display(by_fam.round(5))

In [ ]:
# 11. Download results as one ZIP from Colab.
zipfile_path=OUT.parent/(OUT.name+'_Results.zip')
with zipfile.ZipFile(zipfile_path,'w',zipfile.ZIP_DEFLATED) as z:
 for q in sorted(OUT.iterdir()):
  if q.is_file(): z.write(q,arcname=q.name)
print('Saved archive:',zipfile_path)
try:
 from google.colab import files
 # Uncomment if you want browser download instead of Drive:
 # files.download(str(zipfile_path))
except ImportError:pass